# US Accidents

## 1.Dowmloading Data

In [1]:
!pip install kaggle


   ----- ---------------------------------- 1/8 [tqdm]
   --------------- ------------------------ 3/8 [python-dotenv]
   -------------------- ------------------- 4/8 [mdit-py-plugins]
   ------------------------- -------------- 5/8 [kagglesdk]
   ------------------------- -------------- 5/8 [kagglesdk]
   ------------------------- -------------- 5/8 [kagglesdk]
   ------------------------- -------------- 5/8 [kagglesdk]
   ------------------------- -------------- 5/8 [kagglesdk]
   ------------------------- -------------- 5/8 [kagglesdk]
   ------------------------------ --------- 6/8 [jupytext]
   ------------------------------ --------- 6/8 [jupytext]
   ----------------------------------- ---- 7/8 [kaggle]
   ---------------------------------------- 8/8 [kaggle]




[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

The syntax of the command is incorrect.
'cp' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!kaggle datasets download -d sobhanmoosavi/us-accidents

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


In [4]:
!unzip us-accidents.zip

'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import cudf

cudf.__version__

In [ ]:
df = cudf.read_csv("/content/US_Accidents_March23.csv")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["City"].value_counts()

## 2.Data Cleaning

In [ ]:
df["Timezone"] = df["Timezone"].map(
    {
        "US/Eastern": 0,
        "US/Pacific": 1,
        "US/Central": 2,
        "US/Mountain": 3
    }
)

In [ ]:
df = df.drop(columns=["ID", "Source", "Country", "Description"])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df.to_csv("/content/drive/datasets/accident.csv")

In [ ]:
# df["Start_Time"] = cudf.to_datetime(df["Start_Time"].str[:19])
# df["End_Time"] = cudf.to_datetime(df["End_Time"].str[:19])

In [ ]:
import numpy as np
lens = df["Start_Time"].str.len()

np.unique(lens.values, return_counts=True)

In [ ]:
df["Severity"] = df["Severity"].astype(np.uint8)

In [ ]:
for col in ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]:
    df[col] = df[col].map({"Night": 0, "Day": 1}).astype(np.uint8)

In [ ]:
from sklearn.preprocessing import LabelEncoder

city_encoder = LabelEncoder()

df["City"] = city_encoder.fit_transform(df["City"].to_numpy())
df["City"] = df["City"].astype(np.uint16)

In [ ]:
df.select_dtypes(include="float64").columns

for col in ['Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
            'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']:
    df[col] = df[col].astype(np.int16)

In [ ]:
df[['Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']].describe()

## 3.Aggregation

In [ ]:
pdf_analysis = df[["Severity", "Sunrise_Sunset", "Distance(mi)"]].to_pandas()

In [ ]:
pivot_count = pdf_analysis.pivot_table(
    index="Severity",
    columns="Sunrise_Sunset",
    values="Distance(mi)",
    aggfunc="count"
)
pivot_count.columns = ["Night", "Day"]
pivot_count

## 4.Day/Night Severity

In [ ]:
pivot_percent = (pivot_count / pivot_count.sum(axis=0)) * 100
pivot_percent.round(2)

## 5.Plots

In [ ]:
import matplotlib.pyplot as plt

pivot_count.plot(kind="bar", figsize=(8, 5))
plt.xlabel("شدت تصادف (Severity)")
plt.ylabel("تعداد تصادف")
plt.title("توزیع شدت تصادف بر اساس روز و شب")
plt.legend(["شب", "روز"])
plt.tight_layout()
plt.show()